# 核心概念
浏览器上的内容本质上就是html文件

1. 浏览器输入网址url，发送HTTP 请求到服务
2. 服务器接收请求，返回页面源码（HTML）
3. 浏览器通过解析HTML，执行JS（动态代码），渲染出可视化页面

爬虫：
1. 模拟浏览器，发送HTTP请求，获取HTML源码
2. 解析HTML源码，获取数据
3. 保存数据

## 静态网页抓取

直接发送HTTP请求，服务器返回原始HTML文本，用解析库BeautifulSoup拆分标签、提取文字

但有些时候网站的数据是JS异步动态加载出来的，静态爬虫抓不到，这时必须用动态爬虫

## 动态网页抓取

启动一个真实 Chrome 浏览器进程（无头模式不弹出窗口）
1. 浏览器进程访问 url，加载全部 HTML、自动执行页面 JS 代码
2. 完成数据请求、渲染页面后，再读取渲染完成后的完整页面
3. 再用 BeautifulSoup/selenium 自带方法提取文字数据

In [1]:
# 静态爬虫

import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/catalogue/page-1.html"

# 伪装浏览器头部，防止被拦截
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
}

# 发送网络请求，获取网页原始HTML
resp = requests.get(url, headers=headers)
resp.encoding = "utf-8"

# 将HTML文本转为可解析对象
soup = BeautifulSoup(resp.text, "html.parser")

# 定位所有商品容器
book_boxes = soup.find_all("article", class_="product_pod")

# 循环提取数据
book_list = []
for box in book_boxes:
    book_name = box.h3.a["title"]
    price_text = box.find("p", class_="price_color").text.strip()
    book_list.append({
        "书名": book_name,
        "价格": price_text
    })

print("===== 静态爬虫抓取结果 =====")
for book in book_list[:5]:
    print(book)

===== 静态爬虫抓取结果 =====
{'书名': 'A Light in the Attic', '价格': '£51.77'}
{'书名': 'Tipping the Velvet', '价格': '£53.74'}
{'书名': 'Soumission', '价格': '£50.10'}
{'书名': 'Sharp Objects', '价格': '£47.82'}
{'书名': 'Sapiens: A Brief History of Humankind', '价格': '£54.23'}


In [ ]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

# Edge无头配置
opt = Options()
opt.add_argument("--headless=new")
opt.add_argument("--disable-dev-shm-usage")
opt.add_argument("user-agent=Mozilla/5.0 Edg/120.0.0.0")

# 启动Edge
driver = webdriver.Edge(options=opt)
driver.set_page_load_timeout(10)

try:
    url = "https://www.runoob.com/try/ajax/json_demo.html"
    driver.get(url)
    # 等待JS数据渲染完成
    WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "demo")))

    # 解析页面
    soup = BeautifulSoup(driver.page_source, "html.parser")
    print("===== 动态爬虫抓取结果 =====")
    print(soup.find("div", id="demo").text.strip())
finally:
    driver.quit()

BeautifulSoup 适用于页面结构规整的HTML，可以按 HTML 标签分层拿数据

re 则用于正则匹配，批量捞出所有固定格式的网址